# 10 — Caption Generation

**Marker:** `NOTEBOOK_10_CAPTION_GENERATION_FRESH_V1`

This notebook builds synchronized captions from Notebook 09's exact segment
durations and TTS text.

It produces:

- approximate word-level timing data
- readable phrase-based caption cues
- SRT
- WebVTT
- styled ASS captions for vertical-video burn-in
- a caption manifest for Notebook 11

No speech-recognition model is required.


## Load the project

In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.captions import (
    find_tts_manifest,
    generate_caption_assets,
    load_tts_manifest,
    summarize_caption_manifest,
)

print("NOTEBOOK_10_CAPTION_GENERATION_FRESH_V1")
print(f"Project root: {PROJECT_ROOT}")

NOTEBOOK_10_CAPTION_GENERATION_FRESH_V1
Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

In [2]:
AUDIO_DIRECTORY = PROJECT_ROOT / "data" / "audio"
CAPTION_OUTPUT_DIRECTORY = PROJECT_ROOT / "data" / "captions"

# Leave as None to use the newest tts_manifest.json found recursively.
TTS_MANIFEST_FILENAME = None

CAPTION_STYLE = "phrase"
MAX_WORDS_PER_CUE = 5
MAX_CHARACTERS_PER_LINE = 22
MAX_LINES = 2
MINIMUM_CUE_SECONDS = 0.55
MAXIMUM_CUE_SECONDS = 2.6

# Burn-in appearance used by the generated ASS file.
ASS_FONT_NAME = "Arial"
ASS_FONT_SIZE = 72
ASS_MARGIN_V = 300

print(f"Audio input: {AUDIO_DIRECTORY}")
print(f"Caption output: {CAPTION_OUTPUT_DIRECTORY}")
print(f"Caption style: {CAPTION_STYLE}")

Audio input: c:\Users\hitch\python_files\educational_shorts\data\audio
Caption output: c:\Users\hitch\python_files\educational_shorts\data\captions
Caption style: phrase


## Load the newest TTS manifest

In [3]:
tts_manifest_path = find_tts_manifest(
    audio_directory=AUDIO_DIRECTORY,
    filename=TTS_MANIFEST_FILENAME,
)
tts_manifest = load_tts_manifest(tts_manifest_path)

print(f"TTS manifest: {tts_manifest_path}")
print(f"Topic: {tts_manifest.source_topic_title}")
print(f"Voice: {tts_manifest.voice}")
print(f"Segments: {len(tts_manifest.segments)}")
print(f"Audio duration: {tts_manifest.actual_duration_seconds}s")

TTS manifest: c:\Users\hitch\python_files\educational_shorts\data\audio\what_really_caused_the_boston_tea_party\tts_manifest.json
Topic: What Really Caused the Boston Tea Party?
Voice: am_michael
Segments: 6
Audio duration: 62.452s


## Inspect the exact TTS text

In [4]:
for segment in tts_manifest.segments:
    print(
        f"{segment.index:02d} — {segment.segment_type} "
        f"({segment.actual_seconds:.2f}s)"
    )
    print(segment.tts_text)
    print()

00 — hook (5.87s)
The Boston Tea Party was not just a group of angry colonists throwing tea into the harbor.

01 — what happened (11.53s)
On December 16, seventeen seventy-three, protesters boarded three ships in Boston and dumped 342 chests of East India Company tea into the water.

02 — why they protested (10.21s)
The Tea Act was a factor in the protest, but the main issue was British taxation and trade control, which included the Tea Act as part of broader grievances.

03 — the disguises (12.09s)
Many participants dressed as Mohawk people to avoid identification by British authorities. The sources do not specify what else was damaged or whether the ships were left intact.

04 — historical impact (13.41s)
The Boston Tea Party escalated tensions between Britain and the colonies, contributing to the start of the American Revolution. It played a significant role but was not the single decisive cause of the war.

05 — closing (8.14s)
In one night, the Boston Tea Party intensified tension

## Generate all caption formats

In [5]:
caption_manifest = generate_caption_assets(
    tts_manifest=tts_manifest,
    source_tts_manifest_path=tts_manifest_path,
    output_root=CAPTION_OUTPUT_DIRECTORY,
    caption_style=CAPTION_STYLE,
    max_words_per_cue=MAX_WORDS_PER_CUE,
    max_characters_per_line=MAX_CHARACTERS_PER_LINE,
    max_lines=MAX_LINES,
    minimum_cue_seconds=MINIMUM_CUE_SECONDS,
    maximum_cue_seconds=MAXIMUM_CUE_SECONDS,
    ass_font_name=ASS_FONT_NAME,
    ass_font_size=ASS_FONT_SIZE,
    ass_margin_v=ASS_MARGIN_V,
)

for name, value in summarize_caption_manifest(
    caption_manifest
).items():
    print(f"{name}: {value}")

topic: What Really Caused the Boston Tea Party?
style: phrase
words: 150
cues: 35
average_words_per_cue: 4.29
audio_seconds: 62.452
caption_end_seconds: 62.452
srt: captions.srt
vtt: captions.vtt
ass: captions.ass
output_directory: c:\Users\hitch\python_files\educational_shorts\data\captions\what_really_caused_the_boston_tea_party


## Preview caption cues

In [6]:
for cue in caption_manifest.cues:
    print(
        f"{cue.cue_index:03d} "
        f"{cue.start_seconds:6.2f} --> {cue.end_seconds:6.2f} "
        f"[{cue.segment_type}]"
    )
    print(cue.text)
    print()

001   0.00 -->   1.56 [hook]
The Boston Tea Party
was

002   1.58 -->   2.83 [hook]
not just a group of

003   2.85 -->   4.88 [hook]
angry colonists
throwing tea into

004   4.91 -->   5.87 [hook]
the harbor.

005   6.11 -->   7.43 [what happened]
On December 16,

006   7.45 -->  10.05 [what happened]
seventeen
seventy-three, protesters

007  10.07 -->  12.35 [what happened]
boarded three ships in
Boston

008  12.38 -->  14.31 [what happened]
and dumped 342 chests
of

009  14.33 -->  16.49 [what happened]
East India Company tea
into

010  16.51 -->  17.64 [what happened]
the water.

011  17.88 -->  19.02 [why they protested]
The Tea Act was a

012  19.04 -->  20.54 [why they protested]
factor in the protest,

013  20.57 -->  22.02 [why they protested]
but the main issue was

014  22.04 -->  24.30 [why they protested]
British taxation and
trade control,

015  24.33 -->  25.99 [why they protested]
which included the Tea
Act

016  26.01 -->  28.09 [why they protested]
as part of broader


## Basic timing checks

In [7]:
audio_end = caption_manifest.audio_duration_seconds
caption_end = caption_manifest.final_caption_end_seconds
difference = caption_end - audio_end

print(f"Audio duration: {audio_end:.3f}s")
print(f"Final caption end: {caption_end:.3f}s")
print(f"Difference: {difference:+.3f}s")

overlaps = []

for current, following in zip(
    caption_manifest.cues,
    caption_manifest.cues[1:],
):
    if current.end_seconds > following.start_seconds:
        overlaps.append(
            (current.cue_index, following.cue_index)
        )

print(f"Overlapping cue pairs: {len(overlaps)}")

if abs(difference) > 1.0:
    print(
        "WARNING: Captions end more than one second from the audio end."
    )
else:
    print("Caption ending is close to the narration ending.")

if overlaps:
    print(f"WARNING: overlaps found: {overlaps[:10]}")
else:
    print("No caption overlaps detected.")

Audio duration: 62.452s
Final caption end: 62.452s
Difference: +0.000s
Overlapping cue pairs: 0
Caption ending is close to the narration ending.
No caption overlaps detected.


## Preview generated SRT

In [8]:
caption_directory = Path(
    caption_manifest.output_directory
)
srt_path = caption_directory / caption_manifest.srt_filename

srt_text = srt_path.read_text(encoding="utf-8")
print(srt_text[:4000])

1
00:00:00,000 --> 00:00:01,559
The Boston Tea Party
was

2
00:00:01,584 --> 00:00:02,827
not just a group of

3
00:00:02,852 --> 00:00:04,883
angry colonists
throwing tea into

4
00:00:04,908 --> 00:00:05,868
the harbor.

5
00:00:06,108 --> 00:00:07,427
On December 16,

6
00:00:07,452 --> 00:00:10,046
seventeen
seventy-three, protesters

7
00:00:10,071 --> 00:00:12,351
boarded three ships in
Boston

8
00:00:12,376 --> 00:00:14,306
and dumped 342 chests
of

9
00:00:14,331 --> 00:00:16,489
East India Company tea
into

10
00:00:16,514 --> 00:00:17,642
the water.

11
00:00:17,882 --> 00:00:19,015
The Tea Act was a

12
00:00:19,040 --> 00:00:20,542
factor in the protest,

13
00:00:20,567 --> 00:00:22,018
but the main issue was

14
00:00:22,043 --> 00:00:24,303
British taxation and
trade control,

15
00:00:24,328 --> 00:00:25,986
which included the Tea
Act

16
00:00:26,011 --> 00:00:28,088
as part of broader
grievances.

17
00:00:28,328 --> 00:00:30,595
Many participants
dressed as Mohawk



## Inspect output files

In [9]:
for path in sorted(caption_directory.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")

caption_manifest.json: 9,080 bytes
captions.ass: 3,278 bytes
captions.json: 7,579 bytes
captions.srt: 2,245 bytes
captions.vtt: 2,124 bytes
word_timings.json: 22,901 bytes


## Preview complete caption manifest

In [10]:
print(caption_manifest.model_dump_json(indent=2))

{
  "source_topic_title": "What Really Caused the Boston Tea Party?",
  "source_tts_manifest_filename": "tts_manifest.json",
  "source_audio_filename": "narration.wav",
  "output_directory": "c:\\Users\\hitch\\python_files\\educational_shorts\\data\\captions\\what_really_caused_the_boston_tea_party",
  "caption_json_filename": "captions.json",
  "word_timing_filename": "word_timings.json",
  "srt_filename": "captions.srt",
  "vtt_filename": "captions.vtt",
  "ass_filename": "captions.ass",
  "caption_style": "phrase",
  "max_words_per_cue": 5,
  "max_characters_per_line": 22,
  "max_lines": 2,
  "minimum_cue_seconds": 0.55,
  "maximum_cue_seconds": 2.6,
  "cue_count": 35,
  "word_count": 150,
  "audio_duration_seconds": 62.452,
  "final_caption_end_seconds": 62.452,
  "generated_at_utc": "2026-07-23T16:45:06.832927+00:00",
  "cues": [
    {
      "cue_index": 1,
      "start_seconds": 0.0,
      "end_seconds": 1.559,
      "text": "The Boston Tea Party\nwas",
      "word_count": 5,
   